# NGC 1068 absorbed-corona count scan

This is a fast expected-count calculation for the thermal cutoff-power-law model. The normalization is suppressed by a factor of 50, while the index and pivot remain equal to the current NGC 1068 Case 1 model.

The source response and NGC 1068 60-degree-FoV background are both scaled from 3 to 24 months. No Poisson seed or spectral fit is used. The preferred diagnostic is the support-aware, full COSI data-space Asimov significance

$$Z_A = \sqrt{2\sum_{b_i>0}\left[(s_i+b_i)\ln(1+s_i/b_i)-s_i\right]}.$$

The notebook also reports the simpler total-count $S/\sqrt{S+B}$ value and an energy-binned Asimov value. This is a screening calculation; a final detection claim should use the full likelihood ensemble.

In [ ]:
from pathlib import Path
import sys

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astromodels import Cutoff_powerlaw
from IPython.display import display
from scipy.optimize import brentq

UTILS_DIR = Path(
    '/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/'
    'Sensitivity_calculator'
)
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

from agn_fixed_case_sensitivity import prepare_case

%matplotlib inline

In [ ]:
UNABSORBED_K = 3.1e-1
ABSORBED_FRACTION = 0.02
ABSORBED_K = ABSORBED_FRACTION * UNABSORBED_K
PIVOT_KEV = 1.0
INDEX = -2.10
EXPOSURE_MONTHS = 24
TARGET_SIGMA = 3.0

# The last value is effectively the no-cutoff limit over COSI's band.
CUTOFF_GRID_KEV = np.array([
    50, 75, 100, 128, 150, 200, 300, 400, 500, 750,
    1000, 1500, 2000, 3000, 5000, 10000, 30000, 100000, 1000000,
], dtype=float)

DATA_ROOT = Path(
    '/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/'
    'COSI/Radio_Quiet_AGN'
)
case = {
    'source_name': 'NGC1068',
    'case_tag': 'NGC1068_Case1_absorbed50_count_scan',
    'description': 'NGC 1068 thermal CPL with normalization suppressed by 50',
    'file_stem': 'absorbed50_cutoff_scan',
    'comparison': 'source_detection',
    'longitude_deg': 172.104,
    'latitude_deg': -51.934,
    'fov_cut_deg': 60.0,
    'response_path': str(
        DATA_ROOT / 'DC4_Files' /
        'ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.'
        'filtered.nonsparse.binnedimaging.imagingresponse.h5'
    ),
    'orientation_path': str(
        DATA_ROOT / 'DC4_Files' /
        'DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits'
    ),
    'background_path': str(
        DATA_ROOT / 'DC4_Files/Background' /
        'Total_DC4_BG_3months_binned_data_filtered_with_SAAcut_withSAAbck_'
        'NGC1068_60deg_fov_cut.hdf5'
    ),
    'primary': {
        'kind': 'cpl',
        'K': ABSORBED_K,
        'pivot_keV': PIVOT_KEV,
        'cutoff_keV': 128.0,
        'index': INDEX,
    },
}

print(f'Unabsorbed K: {UNABSORBED_K:g} keV^-1 cm^-2 s^-1 at {PIVOT_KEV:g} keV')
print(f'Absorbed K:   {ABSORBED_K:g} keV^-1 cm^-2 s^-1 ({ABSORBED_FRACTION:g} x)')
print(f'Index: {INDEX:g}; exposure: {EXPOSURE_MONTHS} months; target: {TARGET_SIGMA:g} sigma')

In [ ]:
# prepare_case applies the 60-degree source FoV cut and multiplies the
# three-month source response, background, and livetime by 24/3 = 8.
point_response_cache = {}
prepared = prepare_case(
    case,
    exposure_months=EXPOSURE_MONTHS,
    point_response_cache=point_response_cache,
)
source_response_3m = point_response_cache['injection']
background_24m = prepared['background_expectation']
exposure_multiplier = prepared['exposure_multiplier']

print(f'Exposure multiplier: {exposure_multiplier:g}')
print('Background:', prepared['background_path'])

In [ ]:
def histogram_values(histogram):
    values = histogram.contents
    if hasattr(values, 'compute'):
        values = values.compute()
    if hasattr(values, 'todense'):
        values = values.todense()
    if hasattr(values, 'value'):
        values = values.value
    return np.asarray(values, dtype=float)


def make_absorbed_cpl(cutoff_keV):
    shape = Cutoff_powerlaw()
    shape.K.value = ABSORBED_K
    shape.K.unit = u.keV**-1 * u.cm**-2 * u.s**-1
    shape.piv.value = PIVOT_KEV
    shape.piv.unit = u.keV
    shape.index.value = INDEX
    shape.xc.value = float(cutoff_keV)
    shape.xc.unit = u.keV
    return shape


def poisson_asimov_z(source, background):
    source = np.asarray(source, dtype=float)
    background = np.asarray(background, dtype=float)
    support = background > 0
    term = (source[support] + background[support]) * np.log1p(
        source[support] / background[support]
    ) - source[support]
    return float(np.sqrt(max(0.0, 2.0 * term.sum())))


def required_unabsorbed_fraction(source_at_absorbed_fraction, background):
    def objective(scale_from_absorbed):
        return poisson_asimov_z(
            scale_from_absorbed * source_at_absorbed_fraction, background
        ) - TARGET_SIGMA

    upper = 1.0
    while objective(upper) < 0 and upper < 1e6:
        upper *= 2.0
    if objective(upper) < 0:
        return np.nan
    scale = brentq(objective, 0.0, upper)
    return float(ABSORBED_FRACTION * scale)

In [ ]:
background_cds = histogram_values(background_24m)
background_energy = histogram_values(background_24m.project('Em'))
rows = []

for cutoff_keV in CUTOFF_GRID_KEV:
    source_3m = source_response_3m.get_expectation(
        make_absorbed_cpl(cutoff_keV)
    )
    source_24m = source_3m * exposure_multiplier
    source_24m.axes['Em'].axis_scale = background_24m.axes['Em'].axis_scale
    source_24m = source_24m.to(unit=background_24m.unit, update=False)

    source_cds = histogram_values(source_24m)
    source_energy = histogram_values(source_24m.project('Em'))
    source_counts = float(source_cds.sum())
    background_counts = float(background_cds.sum())
    data_counts = source_counts + background_counts
    total_count_z = source_counts / np.sqrt(data_counts)
    energy_asimov_z = poisson_asimov_z(source_energy, background_energy)
    full_cds_asimov_z = poisson_asimov_z(source_cds, background_cds)
    required_fraction = required_unabsorbed_fraction(source_cds, background_cds)
    background_support = background_cds > 0

    rows.append({
        'cutoff_keV': cutoff_keV,
        'source_counts_24m': source_counts,
        'background_counts_24m': background_counts,
        'data_counts_24m': data_counts,
        'S_over_sqrt_SplusB': total_count_z,
        'energy_binned_Asimov_Z': energy_asimov_z,
        'full_CDS_Asimov_Z': full_cds_asimov_z,
        'required_fraction_of_unabsorbed_K_for_3sigma': required_fraction,
        'equivalent_maximum_suppression_for_3sigma': 1.0 / required_fraction,
        'source_fraction_outside_background_support': (
            source_cds[~background_support].sum() / source_counts
        ),
    })

scan = pd.DataFrame(rows)
display(scan)

In [ ]:
detections = scan.loc[scan['full_CDS_Asimov_Z'] >= TARGET_SIGMA]
maximum_row = scan.loc[scan['full_CDS_Asimov_Z'].idxmax()]

if detections.empty:
    print(
        'No tested cutoff reaches 3 sigma at 0.02 times the unabsorbed normalization.'
    )
    print(
        f"The effectively uncut limit gives Z = {maximum_row['full_CDS_Asimov_Z']:.3f} "
        f"at Ec = {maximum_row['cutoff_keV']:g} keV."
    )
    print(
        'At that cutoff, 3 sigma requires about '
        f"{maximum_row['required_fraction_of_unabsorbed_K_for_3sigma']:.4f} times "
        'the unabsorbed normalization, equivalent to suppression by only '
        f"{maximum_row['equivalent_maximum_suppression_for_3sigma']:.1f}."
    )
else:
    first = detections.iloc[0]
    print(
        f"The first tested cutoff reaching 3 sigma is {first['cutoff_keV']:g} keV "
        f"with Z = {first['full_CDS_Asimov_Z']:.3f}."
    )

In [ ]:
FONT_SIZE = 25
plt.rcParams.update({
    'font.size': FONT_SIZE,
    'font.family': 'Times New Roman',
    'font.weight': '550',
    'axes.linewidth': 1.5,
    'axes.labelweight': '550',
})

fig, ax = plt.subplots(figsize=(12, 9), constrained_layout=True)
ax.xaxis.set_tick_params(
    which='major', size=12, width=1.5, direction='in', top=True,
    pad=8, labelsize=FONT_SIZE,
)
ax.xaxis.set_tick_params(
    which='minor', size=6, width=1.5, direction='in', top=True,
)
ax.yaxis.set_tick_params(
    which='major', size=12, width=1.5, direction='in', right=True,
    pad=8, labelsize=FONT_SIZE,
)
ax.yaxis.set_tick_params(
    which='minor', size=6, width=1.5, direction='in', right=True,
)

ax.plot(
    scan['cutoff_keV'], scan['full_CDS_Asimov_Z'], 'o-', lw=2.5,
    label='Full COSI data-space Asimov',
)
ax.plot(
    scan['cutoff_keV'], scan['energy_binned_Asimov_Z'], 's--', lw=2,
    label='Energy-binned Asimov',
)
ax.plot(
    scan['cutoff_keV'], scan['S_over_sqrt_SplusB'], '^:', lw=2,
    label=r'$S/\sqrt{S+B}$',
)
ax.axhline(TARGET_SIGMA, color='0.2', ls='--', lw=2, label='3 sigma')
ax.set_xscale('log')
ax.set_ylim(bottom=0)
ax.set_xlabel(r'Cutoff energy $E_c$ (keV)', fontsize=FONT_SIZE)
ax.set_ylabel('Expected significance (sigma)', fontsize=FONT_SIZE)
ax.text(
    0.97, 0.97, 'NGC 1068\nThermal CPL, K / 50\n24-month',
    transform=ax.transAxes, ha='right', va='top',
    fontsize=FONT_SIZE, fontweight='550',
)
ax.legend(fontsize=FONT_SIZE, loc='center right', frameon=False)
plt.show()